In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import optuna


from typing import Callable
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay, balanced_accuracy_score, brier_score_loss, roc_auc_score, f1_score, recall_score, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from datetime import date
from enum import Enum
from typing import Callable

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=ConvergenceWarning) 
warnings.simplefilter(action='ignore', category=UserWarning)

ML MODELS

In [ ]:
ODIdataframe  = pd.read_csv('ODIdataframe.csv')
COMIdataframe = pd.read_csv('COMIdataframe.csv')
SF36dataframe = pd.read_csv('SF36dataframe.csv')

In [ ]:
def prepareData(questionnaire: str):
    if questionnaire == "ODI":
        dataframe = ODIdataframe.copy()
        target = "overuse_PreOp_3months_ODI"
        risk = "overuseRiskODI"
    elif questionnaire == "COMI":
        dataframe = COMIdataframe.copy()
        target = "overuse_PreOp_3months_COMI"
        risk = "overuseRiskCOMI"
    elif questionnaire == "SF36":
        dataframe = SF36dataframe.copy()
        target = "overuse_PreOp_3months_SF36"
        risk = "overuseRiskSF36"
    
    dataframe = dataframe.dropna(subset=[target])
    features = [col for col in dataframe.columns if col != target]
    x = dataframe[features]
    y = dataframe[target]

    xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.2, random_state=883)

    riskTest = xTest[risk]

    xTrain.drop(columns=[risk], inplace=True)
    xTest.drop(columns=[risk], inplace=True)

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    
    xTrainScaled = scaler.fit_transform(imputer.fit_transform(xTrain))
    xTestScaled = scaler.transform(imputer.transform(xTest))
    
    return xTrainScaled, xTestScaled, yTrain, yTest, riskTest

In [ ]:
def computeAdditionalMetrics(questionnaire: str, yTrue, yPred, yProb):
    balancedAccuracy = balanced_accuracy_score(yTrue, yPred)
    overallF1 = f1_score(yTrue, yPred, average='macro')
    brierScore = brier_score_loss(yTrue, yProb)
    aucScore = roc_auc_score(yTrue, yProb)

    return balancedAccuracy, overallF1, brierScore, aucScore

LOGISTIC REGRESSION

In [ ]:
def logisticRegression(questionnaire: str, scoringMetric: str | Callable):
    xTrainScaled, xTestScaled, yTrain, yTest, riskTest = prepareData(questionnaire)

    def objective(trial):
        cValue = trial.suggest_float('C', 1e-3, 1e2, log=True)
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
        
        model = LogisticRegression(
            solver='liblinear',
            penalty=penalty,
            C=cValue,
            random_state=883
        )
        
        scores = cross_val_score(
            model, 
            xTrainScaled, 
            yTrain, 
            cv=5, 
            scoring=scoringMetric, 
            n_jobs=-1
        )
        
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100) 

    print(f"Best parameters for {questionnaire}: {study.best_params}")

    # A differenza di GridSearchCV, Optuna non addestra automaticamente il modello finale
    # sull'intero dataset, quindi lo facciamo noi usando i parametri migliori trovati
    logisticModel = LogisticRegression(
        solver='liblinear',
        random_state=883,
        **study.best_params
    )
    logisticModel.fit(xTrainScaled, yTrain)

    prediction = logisticModel.predict(xTestScaled)
    yProbs = logisticModel.predict_proba(xTestScaled)[:, 1]

    return logisticModel, xTestScaled, yTest, prediction, yProbs, \
           confusion_matrix(yTest, prediction), classification_report(yTest, prediction), riskTest

In [ ]:
def evaluateLogisticRegression(questionnaire:str, scoringMetric:str|Callable):
    LogisticModel, xTestScaled, yTest, prediction, yProbs, confusionMatrix, report = logisticRegression(questionnaire, scoringMetric)

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix ({questionnaire})', fontsize=14)

    RocCurveDisplay.from_estimator(
        LogisticModel, 
        xTestScaled, 
        yTest, 
        name="Logistic Regression",
        ax=ax[1],
        curve_kwargs={'color': 'forestgreen'}
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionnaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionnaire, yTest, prediction, yProbs)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
logisticModelODI, xTestScaledODI, yTestODI, predictionODI, yProbsODI, confusionMatrixODI, reportODI = logisticRegression("ODI", "balanced_accuracy")
logisticModelCOMI, xTestScaledCOMI, yTestCOMI, predictionCOMI, yProbsCOMI, confusionMatrixCOMI, reportCOMI = logisticRegression("COMI", "balanced_accuracy")
logisticModelSF36, xTestScaledSF36, yTestSF36, predictionSF36, yProbsSF36, confusionMatrixSF36, reportSF36 = logisticRegression("SF36", "balanced_accuracy")

In [ ]:
evaluateLogisticRegression("ODI", "balanced_accuracy")
evaluateLogisticRegression("COMI", "balanced_accuracy")
evaluateLogisticRegression("SF36", "balanced_accuracy")

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()

models = [
    {"model": logisticModelODI, "x": xTestScaledODI, "y": yTestODI, "label": "Logistic (ODI)", "color": "forestgreen"},
    {"model": logisticModelCOMI, "x": xTestScaledCOMI, "y": yTestCOMI, "label": "Logistic (COMI)", "color": "seagreen"},
    {"model": logisticModelSF36, "x": xTestScaledSF36, "y": yTestSF36, "label": "Logistic (SF36)", "color": "mediumseagreen"}
]


for m in models:
    RocCurveDisplay.from_estimator(
        m["model"], 
        m["x"], 
        m["y"], 
        name=m["label"],
        ax=ax, 
        color=m["color"]
    )

ax.plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)") 
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontsize=15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()

models = [
    {"model": logisticModelODI, "x": xTestScaledODI, "y": yTestODI, "label": "Logistic (ODI)", "color": "forestgreen"},
    {"model": logisticModelCOMI, "x": xTestScaledCOMI, "y": yTestCOMI, "label": "Logistic (COMI)", "color": "blue"},
    {"model": logisticModelSF36, "x": xTestScaledSF36, "y": yTestSF36, "label": "Logistic (SF36)", "color": "orange"}
]


for m in models:
    PrecisionRecallDisplay.from_estimator(
        m["model"], 
        m["x"], 
        m["y"], 
        name=m["label"],
        ax=ax, 
        color=m["color"]
    )

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve Comparison', fontsize=15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.show()

SVM

In [ ]:
def svmModelKernel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest, riskTest = prepareData(questionnaire)

    def objective(trial):
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])
        cValue = trial.suggest_float('C', 0.1, 50.0, log=True)
        
        if kernel == 'linear':
            model = SVC(kernel=kernel, C=cValue, class_weight='balanced', random_state=42, cache_size=1000)
            
        elif kernel == 'rbf':
            gamma = trial.suggest_categorical('gamma_rbf', ['scale', 'auto', 0.1, 0.01])
            model = SVC(kernel=kernel, C=cValue, gamma=gamma, class_weight='balanced', random_state=42, cache_size=1000)
            
        elif kernel == 'poly':
            gamma = trial.suggest_categorical('gamma_poly', ['scale', 'auto'])
            degree = trial.suggest_int('degree', 2, 3)
            model = SVC(kernel=kernel, C=cValue, gamma=gamma, degree=degree, class_weight='balanced', random_state=42, cache_size=1000)

        scores = cross_val_score(model, xTrainScaled, yTrain, cv=5, scoring='balanced_accuracy', n_jobs=-1)
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=80) 

    print(f"Best params: {study.best_params}\n")

    best_params = study.best_params
    
    final_svc_kwargs = {
        'kernel': best_params['kernel'], 
        'C': best_params['C'],
        'class_weight': 'balanced',
        'probability': True,
        'random_state': 42,
        'cache_size': 1000
    }
    
    if best_params['kernel'] == 'rbf':
        final_svc_kwargs['gamma'] = best_params['gamma_rbf']
    elif best_params['kernel'] == 'poly':
        final_svc_kwargs['gamma'] = best_params['gamma_poly']
        final_svc_kwargs['degree'] = best_params['degree']

    bestModel = SVC(**final_svc_kwargs)
    bestModel.fit(xTrainScaled, yTrain)

    prediction = bestModel.predict(xTestScaled)
    yProb = bestModel.predict_proba(xTestScaled)[:, 1]

    return bestModel, xTestScaled, yTest, prediction, yProb, \
           confusion_matrix(yTest, prediction), classification_report(yTest, prediction), riskTest

In [ ]:
def evaluateSVM(questionaire: str):
    model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report, riskTest = svmModelKernel(questionaire)
    modelType = "SVM with kernel"

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix ({modelType})')

    RocCurveDisplay.from_estimator(
        model, 
        xTestScaled, 
        yTest, 
        name=modelType,
        ax=ax[1],
        color='forestgreen'
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionaire, yTest, prediction, yProb)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
evaluateSVM("ODI")

In [ ]:
evaluateSVM("COMI")

In [ ]:
evaluateSVM("SF36")

DECISION TREES (XGBoost)

In [ ]:
def xgboostModel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest, riskTest = prepareData(questionnaire)

    neg_count = np.sum(yTrain == 0)
    pos_count = np.sum(yTrain == 1)
    weight = neg_count / pos_count if pos_count > 0 else 1

    def objective(trial):
        params = {
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'scale_pos_weight': weight,
            'eval_metric': 'logloss',
            'random_state': 42,
            'n_jobs': 1 
        }

        model = XGBClassifier(**params)
        scores = cross_val_score(model, xTrainScaled, yTrain, cv=5, scoring='balanced_accuracy', n_jobs=-1)
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)

    print(f"Best params: {study.best_params}\n")

    bestModel = XGBClassifier(
        scale_pos_weight=weight,
        eval_metric='logloss',
        random_state=42,
        **study.best_params
    )
    bestModel.fit(xTrainScaled, yTrain)

    prediction = bestModel.predict(xTestScaled)
    yProb = bestModel.predict_proba(xTestScaled)[:, 1]
    
    return bestModel, xTestScaled, yTest, prediction, yProb, \
           confusion_matrix(yTest, prediction), classification_report(yTest, prediction), riskTest

In [ ]:
def evaluateXGBoost(questionaire: str):
    model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report, riskTest = xgboostModel(questionaire)

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix - XGBoost')

    RocCurveDisplay.from_estimator(
        model, 
        xTestScaled, 
        yTest, 
        name="XGBoost",
        ax=ax[1],
        color='forestgreen'
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionaire, yTest, prediction, yProb)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
evaluateXGBoost("ODI")

In [ ]:
evaluateXGBoost("COMI")

In [ ]:
evaluateXGBoost("SF36")

RANDOM FOREST

In [ ]:
def randomForestModel(questionnaire: str):
    xTrainScaled, xTestScaled, yTrain, yTest, riskTest = prepareData(questionnaire)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 400),
            'max_depth': trial.suggest_categorical('max_depth', [None, 5, 10, 15, 20]),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'class_weight': 'balanced',
            'random_state': 42,
            'n_jobs': 1 
        }

        model = RandomForestClassifier(**params)
        
        scores = cross_val_score(model, xTrainScaled, yTrain, cv=5, scoring='balanced_accuracy', n_jobs=-1)
        return scores.mean()

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)

    print(f"Best parameters: {study.best_params}\n")

    bestModel = RandomForestClassifier(
        **study.best_params,
        class_weight='balanced',
        random_state=42
    )
    bestModel.fit(xTrainScaled, yTrain)

    prediction = bestModel.predict(xTestScaled)
    yProb = bestModel.predict_proba(xTestScaled)[:, 1]

    return bestModel, xTestScaled, yTest, prediction, yProb, \
           confusion_matrix(yTest, prediction), classification_report(yTest, prediction), riskTest

In [ ]:
def evaluateRandomForest(questionaire: str):
    model, xTestScaled, yTest, prediction, yProb, confusionMatrix, report, riskTest = randomForestModel(questionaire)

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(confusionMatrix, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax[0])
    ax[0].set_xlabel('Predicted Label')
    ax[0].set_ylabel('True Label')
    ax[0].set_title(f'Confusion Matrix - Random Forest')

    RocCurveDisplay.from_estimator(
        model, 
        xTestScaled, 
        yTest, 
        name="Random Forest",
        ax=ax[1],
        color='forestgreen'
    )
    ax[1].plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")
    ax[1].set_title(f"ROC Curve: Predicting Overuse ({questionaire})", fontsize=14)
    ax[1].legend(loc='lower right') 

    plt.tight_layout() 
    plt.show()

    print("\n--- Classification Report ---")
    print(report)

    balancedAccuracy, overallF1, brierScore, aucScore = computeAdditionalMetrics(questionaire, yTest, prediction, yProb)
    print (f"Balanced Accuracy: {balancedAccuracy:.4f}")
    print (f"Overall F1 Score: {overallF1:.4f}")
    print (f"Brier Score: {brierScore:.4f}")
    print (f"AUC Score: {aucScore:.4f}")

In [ ]:
evaluateRandomForest("ODI")

In [ ]:
evaluateRandomForest("COMI")

In [ ]:
evaluateRandomForest("SF36")

totale di 6 modelli, 1 normale, 3 net benefit, 1 WU uno media risultati tau

WEIGHTED UTILITY

In [ ]:
def computeNetBenefit(y_true, y_proba, th=None):
  prop = len(y_true[y_true == 1])/len(y_true)
  y_pred = (y_proba >= th).astype(int)
  sens = recall_score(y_true, y_pred)
  spec = recall_score(y_true, y_pred, pos_label=0)
  return (sens*prop - (1-spec)*(1-prop)*th/(1-th))/prop


def computeWeightedUtility(y_true, y_proba, ths=None, relevances=None):
  if ths is None and relevances is None:
    return computeNetBenefit(y_true, y_proba)
  
  if np.isscalar(ths) and relevances is None:
    return computeNetBenefit(y_true, y_proba, ths)
  
  if relevances is None:
    relevances = np.ones(y_proba.shape)
  
  if ths is None:
    ths = np.ones(y_proba.shape)*0.5
  elif np.isscalar(ths):
    ths = np.ones(y_proba.shape)*ths

  if len(ths) != len(y_true):
    raise ValueError("If not scalar or None, ths should have the same length as y_true")
  if len(relevances) != len(y_true):
    raise ValueError("If not None, relevances should have the same length as y_true")

  pos_idx = y_true == 1
  rs = np.sum(relevances[pos_idx])
  pp = y_proba >= ths
  tp = np.logical_and(pos_idx, pp)
  fp = np.logical_and(np.logical_not(pos_idx), pp)
  return np.sum(tp*relevances)/rs - np.sum(ths/(1-ths)*fp*relevances)/rs

modello net benefit -> un solo ths 
modello wu ths individuale -> interpolazione lineare (vai a cercare la formula)
valutazione

In [ ]:
def evaluateModelNBWU(questionnaire: str, model: Callable):
    logisticModel, xTestScaled, yTest, prediction, yProb, confusionMatrix, report, riskTest = model(questionnaire, "balanced_accuracy")
    netBenefit25 = computeNetBenefit(yTest, yProb, th=0.25)
    netBenefit50 = computeNetBenefit(yTest, yProb, th=0.50)
    netBenefit75 = computeNetBenefit(yTest, yProb, th=0.75)

    thresholds = np.linspace(0.01, 0.99, 100)
    utilities  = [computeNetBenefit(yTest, yProb, th=t) for t in thresholds] # chiedere se è giusto
    netBenefitMedio = np.mean(utilities)

    weightedUtility = computeWeightedUtility(yTest, yProb, riskTest)

    print(f"Net Benefit 0.25: {netBenefit25:.4f}")
    print(f"Net Benefit 0.50: {netBenefit50:.4f}")
    print(f"Net Benefit 0.75: {netBenefit75:.4f}")
    print(f"Average Net Benefit: {netBenefitMedio:.4f}")
    print(f"Weighted utility: {weightedUtility}")

In [ ]:
evaluateModelNBWU("ODI", logisticRegression)

In [ ]:
evaluateModelNBWU("COMI", logisticRegression)

In [ ]:
evaluateModelNBWU("SF36", logisticRegression)

qua interpolazione lineare perchè così per ogni istanza ho un threshold diverso
se ths sempre uguale allora net benefit, se diverso (passi un vettore) allora WU
net benefit 4 valori (soglia fissa 0.25, 0.5, 0.75, media delle soglie)
vogliamo osservare (si spera) ottimizzare per WU da risultati miglior, non si sa per net benefit, meglio se relazione d'ordine balAcc < net ben < WU
aggiungi come metriche WU e netB

LOGISTIC CON NET BENEFIT

In [ ]:
# TODO: sistema codice 1. risk nei modelli 2. ordine del codice 3. feature selection